# GDP Growth Nowcasting

Statistical machine learning coursework project exploring GDP growth prediction using OLS regression, exhaustive best subset selection, and AIC/BIC model selection. The dataset used in this exercise is simulated.


In [ ]:
#Checking files avaibility
import os
print(os.listdir('.'))

['.config', 'ex1_test.csv', 'ex1_train.csv', 'ex1_variables.csv', 'sample_data']


In [ ]:
# Import required libraries for data manipulation, statistical modeling, and evaluation
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
# Load the training, testing, and variable description datasets
train = pd.read_csv("ex1_train.csv")
test = pd.read_csv("ex1_test.csv")
variables = pd.read_csv("ex1_variables.csv")

In [ ]:
# Check the dimensions of the loaded datasets to ensure proper loading
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Variables shape:", variables.shape)

Train shape: (100, 14)
Test shape: (40, 13)
Variables shape: (13, 4)


In [ ]:
# Separate the independent predictor variables (features) from the target variable y
feature_cols = [c for c in train.columns if c not in ['id', 'y']]
y_train = train['y']

In [ ]:
#Add an intercept constant and fit a full OLS regression on all 12 indicators
X_full = sm.add_constant(train[feature_cols])
full_model = sm.OLS(y_train, X_full).fit()

In [ ]:
# Display the full OLS regression summary to examine initial coefficients, R-squared, and p-values
print(full_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.569
Model:                            OLS   Adj. R-squared:                  0.510
Method:                 Least Squares   F-statistic:                     9.586
Date:                Fri, 09 Oct 2026   Prob (F-statistic):           1.43e-11
Time:                        06:32:30   Log-Likelihood:                -75.545
No. Observations:                 100   AIC:                             177.1
Df Residuals:                      87   BIC:                             211.0
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0170      0.055      0.308      0.7

In [ ]:
# Run best subset selection across all 4,095 variable combinations
from itertools import combinations

results = []

for k in range(1, len(feature_cols) + 1):
    for combo in combinations(feature_cols, k):
        X_sub = sm.add_constant(train[list(combo)])
        model_sub = sm.OLS(y_train, X_sub).fit()

        results.append({
            "k": k,
            "features": list(combo),
            "aic": model_sub.aic,
            "bic": model_sub.bic,
            "r2_adj": model_sub.rsquared_adj
        })

res_df = pd.DataFrame(results)

# Confirm that all 4,095 models were evaluated
print("Number of models evaluated:", len(res_df))
print("Best-subset results are ready.")

Number of models evaluated: 4095
Best-subset results are ready.


In [ ]:
# Find the models with the lowest AIC and BIC scores
best_bic_row = res_df.loc[res_df["bic"].idxmin()]
best_aic_row = res_df.loc[res_df["aic"].idxmin()]

In [ ]:
# Print the best feature sets selected by BIC and AIC along with their scores
print("=== BEST BIC MODEL ===")
print(f"Number of features (k): {best_bic_row['k']}")
print(f"Features: {best_bic_row['features']}")
print(f"BIC score: {best_bic_row['bic']:.2f}")

print("\n=== BEST AIC MODEL ===")
print(f"Number of features (k): {best_aic_row['k']}")
print(f"Features: {best_aic_row['features']}")
print(f"AIC score: {best_aic_row['aic']:.2f}")

=== BEST BIC MODEL ===
Number of features (k): 3
Features: ['PAYEMS', 'HOUST', 'GS10TB3Mx']
BIC score: 185.09

=== BEST AIC MODEL ===
Number of features (k): 6
Features: ['PAYEMS', 'HOUST', 'AWHMAN', 'GS10TB3Mx', 'S_P500', 'M2REAL']
AIC score: 169.39


In [ ]:
# Store the feature lists for the Full, AIC, and BIC models
full_features = feature_cols
aic_features = list(best_aic_row["features"])
bic_features = list(best_bic_row["features"])

In [ ]:
# Split the 100 training observations into 75 training rows and 25 validation rows
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    train,
    train["y"],
    test_size=25,
    random_state=42
)

In [ ]:
# Define a function to fit a model on 75 rows and calculate its MSE on 25 held-out rows
def compute_holdout_mse(features):
    X_tr = sm.add_constant(X_train_sub[features])
    X_v = sm.add_constant(X_val[features])

    model = sm.OLS(y_train_sub, X_tr).fit()
    preds = model.predict(X_v)

    return mean_squared_error(y_val, preds)

In [ ]:
# Calculate the holdout MSE for the Full, AIC, and BIC models

mse_full = compute_holdout_mse(full_features)
mse_aic = compute_holdout_mse(aic_features)
mse_bic = compute_holdout_mse(bic_features)

# Print the results
print("=== 75/25 HOLDOUT OUT-OF-SAMPLE MSE RESULTS ===")
print(f"Full Model (12 variables) MSE: {mse_full:.4f}")
print(f"AIC Model (6 variables) MSE: {mse_aic:.4f}")
print(f"BIC Model (3 variables) MSE: {mse_bic:.4f}")

=== 75/25 HOLDOUT OUT-OF-SAMPLE MSE RESULTS ===
Full Model (12 variables) MSE: 0.4837
AIC Model (6 variables) MSE: 0.3463
BIC Model (3 variables) MSE: 0.3295


In [ ]:
# Refit the chosen 3-variable BIC model using all 100 training observations
X_train_full_bic = sm.add_constant(train[bic_features])
final_model = sm.OLS(y_train, X_train_full_bic).fit()

In [ ]:
# Display the regression summary of the final 3-variable BIC model
print(final_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.497
Model:                            OLS   Adj. R-squared:                  0.481
Method:                 Least Squares   F-statistic:                     31.59
Date:                Fri, 09 Oct 2026   Prob (F-statistic):           2.71e-14
Time:                        06:40:24   Log-Likelihood:                -83.334
No. Observations:                 100   AIC:                             174.7
Df Residuals:                      96   BIC:                             185.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0170      0.057      0.299      0.7

In [ ]:
# Add a constant term and generate predictions for the 40 test quarters
X_test_bic = sm.add_constant(test[bic_features])
test_predictions = final_model.predict(X_test_bic)

In [ ]:
# Format the predictions into a two-column DataFrame: id and prediction
submission = pd.DataFrame({
    "id": test["id"],
    "prediction": test_predictions
})
print(submission.head())

          id  prediction
0  test_0001    0.182749
1  test_0002    0.132520
2  test_0003    0.348795
3  test_0004    0.527565
4  test_0005    0.068779


In [ ]:
# Inspect the first 5 rows, total shape, and confirm zero missing values
print("=== FIRST 5 PREDICTIONS ===")
print(submission.head())
print("\nSubmission Shape:", submission.shape)
print("Missing values:", submission.isnull().sum().to_dict())

=== FIRST 5 PREDICTIONS ===
          id  prediction
0  test_0001    0.182749
1  test_0002    0.132520
2  test_0003    0.348795
3  test_0004    0.527565
4  test_0005    0.068779

Submission Shape: (40, 2)
Missing values: {'id': 0, 'prediction': 0}


In [ ]:
# Create the required two-column prediction file
submission = pd.DataFrame({
    "id": test["id"],
    "prediction": test_predictions
})

# Save the file as a CSV
submission.to_csv("ex1_Kumar_Shreyansh.csv", index=False)
print("Prediction CSV saved!")

Prediction CSV saved!
